<a href="https://colab.research.google.com/github/violeo-crypto/compling-hw/blob/main/Naumova_%22fine_tuning_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [2]:
# Установка библиотек
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

# Для работы с GPU (проверяем наличие)
import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


In [3]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
dataset = load_dataset("ag_news")
print("Датасет загружен.")
print(f"Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")
print(f"Метки классов: {dataset['train'].features['label'].names}")  # 4 класса

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен.
Train: 120000, Test: 7600
Метки классов: ['World', 'Sports', 'Business', 'Sci/Tech']


In [5]:
dataset['test'][0]

{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.",
 'label': 2}

In [7]:
#Загрузка модели и токенизатора
model_name = "distilbert-base-uncased"  # или "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4  # 4 категории новостей
).to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# Токенизация данных
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [9]:
# accuracy
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [10]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./ag_news_results",      # папка для сохранения чекпоинтов
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",               # оцениваем после каждой эпохи
    save_strategy="epoch",                # сохраняем после каждой эпохи
    load_best_model_at_end=True,          # загружаем лучшую модель по окончании
    metric_for_best_model="accuracy",     # метрика для выбора лучшей модели
    report_to="none",                      # отключаем логирование (можно включить tensorboard)
)

In [11]:
# Создание тренера
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Начинаем обучение...")
trainer.train()

Начинаем обучение...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.198976,0.178306,0.941842
2,0.137239,0.186607,0.948684
3,0.092340,0.216937,0.945658


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=22500, training_loss=0.15403148142496745, metrics={'train_runtime': 3322.5392, 'train_samples_per_second': 108.351, 'train_steps_per_second': 6.772, 'total_flos': 8558812764910464.0, 'train_loss': 0.15403148142496745, 'epoch': 3.0})

In [12]:
# Оценка
eval_results = trainer.evaluate()
print(f"\nAccuracy на тесте: {eval_results['eval_accuracy']:.4f}")


Accuracy на тесте: 0.9487


In [13]:
# Сохранение модели в папку
model.save_pretrained("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")
print("Модель сохранена в папку ./ag_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в папку ./ag_news_model


In [15]:
# тест модели на трех новых новостях
from transformers import pipeline

# Загружаем сохранённую модель через pipeline
classifier = pipeline(
    "text-classification",
    model="./ag_news_model",
    tokenizer="./ag_news_model"
)

# Список меток (сопоставление индексов с названиями)
label_names = dataset["train"].features["label"].names  # ['World', 'Sports', 'Business', 'Sci/Tech']

# Три новости для проверки
new_news = [
    "Rescuers blame weather and 'underprepared skiers' for rise in Alps avalanche deaths.",
    "Aryna Sabalenka and Elena Rybakina will meet in a final for the third time in just over four months at Indian Wells on Sunday.",
    "Firms are working to make the motors that drive robots more efficient and cheaper."
]

print("\nРезультаты на новых новостях:")
for text in new_news:
    result = classifier(text)[0]
    label_index = int(result["label"].split("_")[-1])
    predicted_class = label_names[label_index]
    confidence = result["score"]
    print(f"Текст: {text}")
    print(f"Предсказанный класс: {predicted_class}, уверенность: {confidence:.4f}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Результаты на новых новостях:
Текст: Rescuers blame weather and 'underprepared skiers' for rise in Alps avalanche deaths.
Предсказанный класс: World, уверенность: 0.8620

Текст: Aryna Sabalenka and Elena Rybakina will meet in a final for the third time in just over four months at Indian Wells on Sunday.
Предсказанный класс: Sports, уверенность: 0.9981

Текст: Firms are working to make the motors that drive robots more efficient and cheaper.
Предсказанный класс: Sci/Tech, уверенность: 0.9595

